# Ablation 5 — Segmentation Query Sensitivity

**Thesis Table**: Section 5.x — Text query influence on segmentation

Tests different natural-language query strings on the same signal to understand
how sensitive the MLP segmentation is to the query embedding.

Signals tested: the three Reactor 1 labelled signals.

Queries per signal:
- **Drift**: correct / vague / wrong-class / no-query (zero vector)
- **Frozen**: correct / vague / wrong-class / no-query
- **Spikes**: correct / vague / wrong-class / no-query

Key finding to replicate: the heatmap from Cell 116 of original notebook showing
F1 as a function of (signal × query) — query specificity matters most for drift,
least for frozen (flat sensor is recognizable from time-series alone).

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.data.timeseer_client import fetch_series_api
from src.models.chatts_loader import load_model
from src.models.mlp import load_mlp
from src.inference.embeddings import encode_text_query
from src.inference.chronos_server import start_server, get_chronos_embedding_cached, shutdown_server
from src.preprocessing.chunking import downsample
from src.segmentation.pipeline import segment_signal

import os
VSC_SCRATCH = os.environ.get('VSC_SCRATCH', '/scratch/leuven/375/vsc37531')
MLP_PATH    = os.path.join(VSC_SCRATCH, 'ChatTS', 'timeseer_data', 'segmentation_mlp_final.pt')
THRESHOLD   = 0.65
WINDOW_SIZE = 32  # best single-scale window from ab1

print('Imports OK')

In [ ]:
model, tokenizer, processor = load_model('ChatTS-14B')
def _encode(q): return encode_text_query(q, model=model, tokenizer=tokenizer)

start_server()
mlp = load_mlp(MLP_PATH)

In [ ]:
# Signal definitions and ground-truth ranges (original 8640-pt space)
TEST_SIGNALS = [
    ('R1-AT-101-PH',   'Reactor 1', 'drift',   0,    8640),
    ('R1-AT-103-DO',   'Reactor 1', 'frozen',  2131, 2259),
    ('R1-AT-102-COND', 'Reactor 1', 'spikes',  1460, 1475),
]

# Queries per anomaly type: (name, query_string)
QUERIES = {
    'drift': [
        ('correct',     'Find gradual baseline drift'),
        ('vague',       'Find periods with abnormal dynamics'),
        ('wrong_class', 'Detect stale data or frozen sensor readings'),
        ('zero',        None),  # None → zero embedding
    ],
    'frozen': [
        ('correct',     'Detect stale data'),
        ('vague',       'Find periods with abnormal dynamics'),
        ('wrong_class', 'Find gradual baseline drift'),
        ('zero',        None),
    ],
    'spikes': [
        ('correct',     'Identify temperature spikes'),
        ('vague',       'Find periods with abnormal dynamics'),
        ('wrong_class', 'Find gradual baseline drift'),
        ('zero',        None),
    ],
}

print('Configuration OK')

In [ ]:
results = {}  # (tag, query_name) → f1

for tag, area, anom_type, gt_start, gt_end in TEST_SIGNALS:
    print(f'\n── {tag} ({anom_type}) ──')
    vals, idx = fetch_series_api(tag, area)
    vals_ds   = downsample(vals, target=1024)
    n = len(vals_ds)

    true_s = int(gt_start * n / len(vals))
    true_e = int(gt_end   * n / len(vals))
    true_mask = np.zeros(n)
    true_mask[true_s:true_e] = 1.0

    for qname, qtext in QUERIES[anom_type]:
        if qtext is None:
            # Zero-vector query — bypass encode_text_query
            embed_fn = lambda q: np.zeros(5120, dtype=np.float32)
            prob_map, _ = segment_signal(
                vals_ds, 'placeholder', mlp, embed_fn,
                get_chronos_embedding_cached,
                window_size=WINDOW_SIZE, threshold=THRESHOLD
            )
        else:
            prob_map, _ = segment_signal(
                vals_ds, qtext, mlp, _encode,
                get_chronos_embedding_cached,
                window_size=WINDOW_SIZE, threshold=THRESHOLD
            )

        binary = (prob_map > THRESHOLD).astype(float)
        tp = ((binary == 1) & (true_mask == 1)).sum()
        fp = ((binary == 1) & (true_mask == 0)).sum()
        fn = ((binary == 0) & (true_mask == 1)).sum()
        prec = tp / (tp + fp + 1e-8)
        rec  = tp / (tp + fn + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)
        results[(tag, qname)] = round(float(f1), 3)
        print(f'  {qname:<14}: F1={f1:.3f}  TP={int(tp)}  FP={int(fp)}  FN={int(fn)}')

In [ ]:
# Print thesis-ready table
query_names = ['correct', 'vague', 'wrong_class', 'zero']

print(f'\n{"Signal":<22} {"Type":<8}', end='')
for qn in query_names:
    print(f'  {qn:<12}', end='')
print()
print('-'*70)

for tag, area, anom_type, _, _ in TEST_SIGNALS:
    print(f'{tag:<22} {anom_type:<8}', end='')
    for qn in query_names:
        f1 = results.get((tag, qn), 0.0)
        print(f'  {f1:.3f}       ', end='')
    print()

shutdown_server()

In [ ]:
# Heatmap visualisation (replicates Cell 116 of original notebook)
query_names = ['correct', 'vague', 'wrong_class', 'zero']
signal_labels = [t[0] for t in TEST_SIGNALS]

matrix = np.array([
    [results.get((tag, qn), 0.0) for qn in query_names]
    for tag, *_ in TEST_SIGNALS
])

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(query_names)))
ax.set_xticklabels(query_names, rotation=25, ha='right')
ax.set_yticks(range(len(signal_labels)))
ax.set_yticklabels(signal_labels)
ax.set_title('Segmentation F1 by Signal × Query Type (w=32, threshold=0.65)')

for i in range(len(signal_labels)):
    for j in range(len(query_names)):
        v = matrix[i, j]
        color = 'white' if v < 0.4 else 'black'
        ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                fontsize=11, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='F1 score')
plt.tight_layout()
plt.savefig('../../data/figures/ab5_query_sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved heatmap.')